In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import time

In [3]:
# 1. Dataset Preparation
TRAIN_DIR = 'dataset/train'
TEST_DIR = 'dataset/test'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

# Normalize pixel values
normalization_layer = tf.keras.layers.Rescaling(1./255.)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

Found 8000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [4]:
# 2. Function to build models
def build_model(base_model_class, name):
    # Load base model with ImageNet weights, exclude top
    base = base_model_class(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = False  # Freeze base layers
    
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)
    # Binary classification output (1 neuron, sigmoid)
    predictions = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base.input, outputs=predictions)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# 3. Train and Evaluate Loop
results = []
architectures = [
    (MobileNetV2, 'MobileNetV2'),
    (ResNet50, 'ResNet50'),
    (EfficientNetB0, 'EfficientNetB0')
]

for arch, name in architectures:
    print(f"\n--- Training {name} ---")
    model = build_model(arch, name)
    
    start_time = time.time()
    history = model.fit(train_ds, epochs=5, validation_data=test_ds, verbose=1)
    end_time = time.time()
    
    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    duration = end_time - start_time
    
    results.append({
        'Model': name,
        'Train Acc': f"{train_acc:.2%}",
        'Val Acc': f"{val_acc:.2%}",
        'Time (s)': f"{duration:.1f}"
    })
    
    # Save the best model for Streamlit
    if name == 'MobileNetV2': # Assuming this wins
        model.save('best_model.h5')

print("\nTraining Complete.")


--- Training MobileNetV2 ---
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 177s 659ms/step - accuracy: 0.9514 - loss: 0.1343 - val_accuracy: 0.9790 - val_loss: 0.0590
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 156s 624ms/step - accuracy: 0.9804 - loss: 0.0544 - val_accuracy: 0.9820 - val_loss: 0.0497
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 158s 631ms/step - accuracy: 0.9833 - loss: 0.0446 - val_accuracy: 0.9830 - val_loss: 0.0460
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 197s 610ms/step - accuracy: 0.9847 - loss: 0.0407 - val_accuracy: 0.9845 - val_loss: 0.0457
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 155s 617ms/step - accuracy: 0.9868 - loss: 0.0361 - val_accuracy: 0.9860 - val_loss: 0.0440

--- Training ResNet50 ---
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 617s 2s/step - accuracy: 0.5390 - loss: 0.6954 - val_accuracy: 0.5965 - val_loss: 0.6665
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 592s 2s/step - accuracy: 0.6015 - loss: 0.6606 - val_accuracy: 0.6185 - val_loss: 0.6583
Epoch 3/5
250/250 ━━━━━━━━━━━━━━


Training Complete.
